In [27]:
import sqlite3
conn = sqlite3.connect("flightdb.db")  # new database file
cursor = conn.cursor()


In [6]:
#creating a database ->flightdb

import sqlite3

# Create database
conn = sqlite3.connect("flightdb.db")
cursor = conn.cursor()

# Drop table if it somehow exists to ensure a clean slate
cursor.execute("DROP TABLE IF EXISTS airport;")

# Create tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS airport (
    airport_id INTEGER PRIMARY KEY AUTOINCREMENT,
    icao_code TEXT UNIQUE,
    iata_code TEXT UNIQUE,
    name TEXT,
    city TEXT,
    country TEXT,
    continent TEXT,
    latitude REAL,
    longitude REAL,
    timezone TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS aircraft (
    aircraft_id INTEGER PRIMARY KEY AUTOINCREMENT,
    registration TEXT UNIQUE,
    model TEXT,
    manufacturer TEXT,
    icao_type_code TEXT,
    owner TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS flights (
    flight_id TEXT PRIMARY KEY,
    flight_number TEXT,
    aircraft_registration TEXT,
    origin_iata TEXT,
    destination_iata TEXT,
    scheduled_departure TEXT,
    actual_departure TEXT,
    scheduled_arrival TEXT,
    actual_arrival TEXT,
    status TEXT,
    airline_code TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS airport_delays (
    delay_id INTEGER PRIMARY KEY AUTOINCREMENT,
    airport_iata TEXT,
    delay_date TEXT,
    total_flights INTEGER,
    delayed_flights INTEGER,
    avg_delay_min INTEGER,
    median_delay_min INTEGER,
    canceled_flights INTEGER
)
""")

conn.commit()


In [23]:
#Fetching Airport data using the Aerodatabox API
import requests
import time

headers = {
   "x-rapidapi-key": "c07681eddemsh68bef06446b6e44p1c266ajsncd7de52d3ae4",
	"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	"Content-Type": "application/json"
}

# List of airports
airport_codes = ["LHR", "DEL", "BLR", "BOM", "HYD", "MAA", "JFK", "SYD", "SIN", "LAX"]

# Loop through each airport
for code in airport_codes:
    url = f"https://aerodatabox.p.rapidapi.com/airports/iata/{code}"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        print(f"{code}: {response.json()}")
    else:
        print(f"{code}: Error {response.status_code} - {response.text}")
    
    
    time.sleep(2)  # wait 2 seconds before next request to avoid rate limits


LHR: {'icao': 'EGLL', 'iata': 'LHR', 'shortName': 'Heathrow', 'fullName': 'London Heathrow', 'municipalityName': 'London', 'location': {'lat': 51.4706, 'lon': -0.461941}, 'elevation': {'meter': 25.3, 'km': 0.03, 'mile': 0.02, 'nm': 0.01, 'feet': 83.0}, 'country': {'code': 'GB', 'name': 'United Kingdom'}, 'continent': {'code': 'EU', 'name': 'Europe'}, 'timeZone': 'Europe/London', 'urls': {'webSite': 'http://www.heathrow.com/', 'wikipedia': 'https://en.wikipedia.org/wiki/London_Heathrow_Airport', 'twitter': 'https://x.com/HeathrowAirport', 'flightRadar': 'https://www.flightradar24.com/51.47,-0.46/14', 'googleMaps': 'https://www.google.com/maps/@51.470600,-0.461941,14z'}}
DEL: {'icao': 'VIDP', 'iata': 'DEL', 'shortName': 'Indira Gandhi', 'fullName': 'New Delhi Indira Gandhi', 'municipalityName': 'New Delhi', 'location': {'lat': 28.5665, 'lon': 77.1031}, 'elevation': {'meter': 236.83, 'km': 0.24, 'mile': 0.15, 'nm': 0.13, 'feet': 777.0}, 'country': {'code': 'IN', 'name': 'India'}, 'contine

In [1]:
#Inserting Airport data into SQLite database
import requests
import sqlite3
import time

# Connect to SQLite
conn_sql = sqlite3.connect("flightdb.db")
cursor = conn_sql.cursor()

# Airport codes list
airport_codes = ["BLR", "BOM", "DEL", "MAA", "HYD", "JFK", "LHR", "SIN", "SYD", "LAX"]

# API headers
headers = {
    "x-rapidapi-key": "c07681eddemsh68bef06446b6e44p1c266ajsncd7de52d3ae4",  
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
}

# Loop through each airport code
for code in airport_codes:
    url = f"https://aerodatabox.p.rapidapi.com/airports/iata/{code}"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        airport_data = response.json()

        # Map fields to table (use shortName/fullName instead of name to avoid NULLs)
        airport_info = (
            airport_data.get("icao"),
            airport_data.get("iata"),
            airport_data.get("shortName") or airport_data.get("fullName"),
            airport_data.get("municipalityName") or airport_data.get("location", {}).get("city"),
            airport_data.get("location", {}).get("country"),
            airport_data.get("location", {}).get("continent"),
            airport_data.get("location", {}).get("lat"),
            airport_data.get("location", {}).get("lon"),
            airport_data.get("timezone")
        )

        # Insert into SQLite
        cursor.execute("""
        INSERT OR IGNORE INTO airport 
        (icao_code, iata_code, name, city, country, continent, latitude, longitude, timezone)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, airport_info)

        print(f"Inserted airport: {code}")
    else:
        print(f"{code}: Error {response.status_code} - {response.text}")

    time.sleep(2)  # wait 2 seconds before next request

conn_sql.commit()

Inserted airport: BLR
Inserted airport: BOM
Inserted airport: DEL
Inserted airport: MAA
Inserted airport: HYD
Inserted airport: JFK
Inserted airport: LHR
Inserted airport: SIN
Inserted airport: SYD
Inserted airport: LAX


In [2]:
#inserting Airport delay data into SQLite database

import requests
import sqlite3
import time

# Connect to SQLite
conn_sql = sqlite3.connect("flightdb.db")
cursor = conn_sql.cursor()

airport_codes = ["BLR", "BOM", "DEL", "MAA", "HYD", "JFK", "LHR", "SIN", "SYD", "LAX"]

headers = {
    "x-rapidapi-key": "c07681eddemsh68bef06446b6e44p1c266ajsncd7de52d3ae4",  
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
}

for code in airport_codes:
    url = f"https://aerodatabox.p.rapidapi.com/airports/iata/{code}/delays"
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"{code}: Error {response.status_code} - {response.text}")
        time.sleep(2)
        continue

    delay_data = response.json()

    # Skip if API returned an error message
    if "message" in delay_data:
        print(f"{code}: {delay_data['message']}")
        time.sleep(2)
        continue

    departures = delay_data.get("departuresDelayInformation", {})
    delay_date = delay_data.get("from", {}).get("utc")

    # Convert medianDelay "HH:MM:SS" → minutes
    median_str = departures.get("medianDelay", "00:00:00")
    h, m, s = [int(x) for x in median_str.split(":")]
    median_minutes = h * 60 + m + (s // 60)

    # Approximate avg delay from delayIndex (ratio × 60 minutes)
    avg_delay_minutes = int(departures.get("delayIndex", 0) * 60)

    delay_info = (
        code,                          # airport_iata
        delay_date,                    # delay_date
        departures.get("numTotal"),    # total_flights
        departures.get("numQualifiedTotal"),  # delayed_flights
        avg_delay_minutes,             # avg_delay_min
        median_minutes,                # median_delay_min
        departures.get("numCancelled") # canceled_flights
    )

    cursor.execute("""
    INSERT INTO airport_delays
    (airport_iata, delay_date, total_flights, delayed_flights, avg_delay_min, median_delay_min, canceled_flights)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, delay_info)

    print(f"Inserted delays for airport: {code}")
    time.sleep(2)  # avoid rate limit

conn_sql.commit()

Inserted delays for airport: BLR
Inserted delays for airport: BOM
Inserted delays for airport: DEL
MAA: Error 400 - {"message":"Delay information is unavailable for this airport. Live flight information for this airport is not available or or obsolete."}
Inserted delays for airport: HYD
Inserted delays for airport: JFK
Inserted delays for airport: LHR
Inserted delays for airport: SIN
Inserted delays for airport: SYD
Inserted delays for airport: LAX


In [3]:
# Corrected Cell 5: Inserting aircraft data for first 12 hours
import requests
import sqlite3
import time

conn_sql = sqlite3.connect("flightdb.db")
cursor = conn_sql.cursor()

airport_codes = ["BLR", "BOM", "DEL", "MAA", "HYD", "JFK", "LHR", "SIN", "SYD", "LAX"]

headers = {
    "x-rapidapi-key": "c07681eddemsh68bef06446b6e44p1c266ajsncd7de52d3ae4",  
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
}

def safe_value(val):
    if isinstance(val, dict):
        return val.get("utc") or val.get("local")
    return val

for code in airport_codes:
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/{code}/2026-04-04T00:00/2026-04-04T12:00"
    params = {"withLeg": "true", "direction": "Both", "withCancelled": "true", "withCodeshared": "true", "withCargo": "true", "withPrivate": "true", "withLocation": "false"}

    response = requests.get(url, headers=headers, params=params)
    if response.status_code != 200:
        time.sleep(2)
        continue

    flights_data = response.json()

    # 1. PROCESS DEPARTURES (Origin is always 'code')
    for flight in flights_data.get("departures", []):
        departure = flight.get("departure", {})
        arrival = flight.get("arrival", {})
        aircraft = flight.get("aircraft", {})
        airline = flight.get("airline", {})

        origin_iata = code  
        destination_iata = arrival.get("iata") or arrival.get("airport", {}).get("iata")

        flight_info = (flight.get("number") if flight.get("id") is None else flight.get("id"), flight.get("number"), aircraft.get("reg"), origin_iata, destination_iata, safe_value(departure.get("scheduledTime")), safe_value(departure.get("actualTime")), safe_value(arrival.get("scheduledTime")), safe_value(arrival.get("actualTime")), flight.get("status"), airline.get("iata"))
        cursor.execute("INSERT OR IGNORE INTO flights VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", flight_info)

    # 2. PROCESS ARRIVALS (Destination is always 'code')
    for flight in flights_data.get("arrivals", []):
        departure = flight.get("departure", {})
        arrival = flight.get("arrival", {})
        aircraft = flight.get("aircraft", {})
        airline = flight.get("airline", {})

        origin_iata = departure.get("iata") or departure.get("airport", {}).get("iata")
        destination_iata = code  

        flight_info = (flight.get("number") if flight.get("id") is None else flight.get("id"), flight.get("number"), aircraft.get("reg"), origin_iata, destination_iata, safe_value(departure.get("scheduledTime")), safe_value(departure.get("actualTime")), safe_value(arrival.get("scheduledTime")), safe_value(arrival.get("actualTime")), flight.get("status"), airline.get("iata"))
        cursor.execute("INSERT OR IGNORE INTO flights VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", flight_info)

    print(f"Inserted flights accurately for airport: {code}")
    time.sleep(2)

conn_sql.commit()

Inserted flights accurately for airport: BLR
Inserted flights accurately for airport: BOM
Inserted flights accurately for airport: DEL
Inserted flights accurately for airport: MAA
Inserted flights accurately for airport: HYD
Inserted flights accurately for airport: JFK
Inserted flights accurately for airport: LHR
Inserted flights accurately for airport: SIN
Inserted flights accurately for airport: SYD
Inserted flights accurately for airport: LAX


In [ ]:
#Inserting Aircraft data for first 12 hours into SQLite database

import requests
import sqlite3
import time

# Connect to SQLite
conn_sql = sqlite3.connect("flightdb.db")
cursor = conn_sql.cursor()

# Airport codes list
airport_codes = ["BLR", "BOM", "DEL", "MAA", "HYD", "JFK", "LHR", "SIN", "SYD", "LAX"]

# API headers
headers = {
    "x-rapidapi-key": "c07681eddemsh68bef06446b6e44p1c266ajsncd7de52d3ae4",  
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
}

def safe_value(val):
    if isinstance(val, dict):
        return val.get("utc") or val.get("local")
    return val

# Loop through each airport code
for code in airport_codes:
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/{code}/2026-04-04T00:00/2026-04-04T12:00"
    params = {
        "withLeg": "true",
        "direction": "Both",
        "withCancelled": "true",
        "withCodeshared": "true",
        "withCargo": "true",
        "withPrivate": "true",
        "withLocation": "false"
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        print(f"{code}: Error {response.status_code} - {response.text}")
        time.sleep(2)
        continue

    flights_data = response.json()

    # Insert departures + arrivals for this airport
    for flight in flights_data.get("departures", []) + flights_data.get("arrivals", []):
        departure = flight.get("departure", {})
        arrival = flight.get("arrival", {})
        aircraft = flight.get("aircraft", {})
        airline = flight.get("airline", {})

        # Handle nested airport dictionaries
        origin_iata = departure.get("iata") or departure.get("airport", {}).get("iata")
        destination_iata = arrival.get("iata") or arrival.get("airport", {}).get("iata")

        flight_info = (
            flight.get("id"),
            flight.get("number"),
            aircraft.get("reg"),
            origin_iata,
            destination_iata,
            safe_value(departure.get("scheduledTime")),
            safe_value(departure.get("actualTime")),
            safe_value(arrival.get("scheduledTime")),
            safe_value(arrival.get("actualTime")),
            flight.get("status"),
            airline.get("iata")
        )

        cursor.execute("""
        INSERT OR IGNORE INTO flights
        (flight_id, flight_number, aircraft_registration, origin_iata, destination_iata,
         scheduled_departure, actual_departure, scheduled_arrival, actual_arrival, status, airline_code)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, flight_info)

    print(f"Inserted flights for airport: {code}")
    time.sleep(2)  # avoid API rate limits

conn_sql.commit()

Inserted flights for airport: BLR
Inserted flights for airport: BOM
Inserted flights for airport: DEL
Inserted flights for airport: MAA
Inserted flights for airport: HYD
Inserted flights for airport: JFK
Inserted flights for airport: LHR
Inserted flights for airport: SIN
Inserted flights for airport: SYD
Inserted flights for airport: LAX


In [4]:
# Corrected Cell 6: Inserting Flight data for next 12 hours
for code in airport_codes:
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/{code}/2026-04-04T12:00/2026-04-04T23:59"
    params = {"withLeg": "true", "direction": "Both", "withCancelled": "true", "withCodeshared": "true", "withCargo": "true", "withPrivate": "true", "withLocation": "false"}

    response = requests.get(url, headers=headers, params=params)
    if response.status_code != 200:
        time.sleep(2)
        continue

    flights_data = response.json()

    # 1. PROCESS DEPARTURES
    for flight in flights_data.get("departures", []):
        departure = flight.get("departure", {})
        arrival = flight.get("arrival", {})
        aircraft = flight.get("aircraft", {})
        airline = flight.get("airline", {})

        origin_iata = code
        destination_iata = arrival.get("iata") or arrival.get("airport", {}).get("iata")

        flight_info = (flight.get("number") if flight.get("id") is None else flight.get("id"), flight.get("number"), aircraft.get("reg"), origin_iata, destination_iata, safe_value(departure.get("scheduledTime")), safe_value(departure.get("actualTime")), safe_value(arrival.get("scheduledTime")), safe_value(arrival.get("actualTime")), flight.get("status"), airline.get("iata"))
        cursor.execute("INSERT OR IGNORE INTO flights VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", flight_info)

    # 2. PROCESS ARRIVALS
    for flight in flights_data.get("arrivals", []):
        departure = flight.get("departure", {})
        arrival = flight.get("arrival", {})
        aircraft = flight.get("aircraft", {})
        airline = flight.get("airline", {})

        origin_iata = departure.get("iata") or departure.get("airport", {}).get("iata")
        destination_iata = code

        flight_info = (flight.get("number") if flight.get("id") is None else flight.get("id"), flight.get("number"), aircraft.get("reg"), origin_iata, destination_iata, safe_value(departure.get("scheduledTime")), safe_value(departure.get("actualTime")), safe_value(arrival.get("scheduledTime")), safe_value(arrival.get("actualTime")), flight.get("status"), airline.get("iata"))
        cursor.execute("INSERT OR IGNORE INTO flights VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", flight_info)

    print(f"Inserted next 12h flights accurately for airport: {code}")
    time.sleep(2)

conn_sql.commit()

Inserted next 12h flights accurately for airport: BLR
Inserted next 12h flights accurately for airport: BOM
Inserted next 12h flights accurately for airport: DEL
Inserted next 12h flights accurately for airport: MAA
Inserted next 12h flights accurately for airport: HYD
Inserted next 12h flights accurately for airport: JFK
Inserted next 12h flights accurately for airport: LHR
Inserted next 12h flights accurately for airport: SIN
Inserted next 12h flights accurately for airport: SYD
Inserted next 12h flights accurately for airport: LAX


In [ ]:
#Inserting Flight data arrival and departure for next 12 hours into SQLite database

import requests
import sqlite3
import time

# Connect to SQLite
conn_sql = sqlite3.connect("flightdb.db")
cursor = conn_sql.cursor()

# Airport codes list
airport_codes = ["BLR", "BOM", "DEL", "MAA", "HYD", "JFK", "LHR", "SIN", "SYD", "LAX"]

# API headers
headers = {
    "x-rapidapi-key": "c07681eddemsh68bef06446b6e44p1c266ajsncd7de52d3ae4",  
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
}

def safe_value(val):
    if isinstance(val, dict):
        return val.get("utc") or val.get("local")
    return val

# Loop through each airport code
for code in airport_codes:
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/{code}/2026-04-04T12:00/2026-04-04T23:59"
    params = {
        "withLeg": "true",
        "direction": "Both",
        "withCancelled": "true",
        "withCodeshared": "true",
        "withCargo": "true",
        "withPrivate": "true",
        "withLocation": "false"
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        print(f"{code}: Error {response.status_code} - {response.text}")
        time.sleep(2)
        continue

    flights_data = response.json()

    # Insert departures + arrivals for this airport
    for flight in flights_data.get("departures", []) + flights_data.get("arrivals", []):
        departure = flight.get("departure", {})
        arrival = flight.get("arrival", {})
        aircraft = flight.get("aircraft", {})
        airline = flight.get("airline", {})

        # Handle nested airport dictionaries
        origin_iata = departure.get("iata") or departure.get("airport", {}).get("iata")
        destination_iata = arrival.get("iata") or arrival.get("airport", {}).get("iata")

        flight_info = (
            flight.get("id"),
            flight.get("number"),
            aircraft.get("reg"),
            origin_iata,
            destination_iata,
            safe_value(departure.get("scheduledTime")),
            safe_value(departure.get("actualTime")),
            safe_value(arrival.get("scheduledTime")),
            safe_value(arrival.get("actualTime")),
            flight.get("status"),
            airline.get("iata")
        )

        cursor.execute("""
        INSERT OR IGNORE INTO flights
        (flight_id, flight_number, aircraft_registration, origin_iata, destination_iata,
         scheduled_departure, actual_departure, scheduled_arrival, actual_arrival, status, airline_code)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, flight_info)

    print(f"Inserted flights for airport: {code}")
    time.sleep(2)  # avoid API rate limits

conn_sql.commit()

Inserted flights for airport: BLR
Inserted flights for airport: BOM
Inserted flights for airport: DEL
Inserted flights for airport: MAA
Inserted flights for airport: HYD
Inserted flights for airport: JFK
Inserted flights for airport: LHR
Inserted flights for airport: SIN
Inserted flights for airport: SYD
Inserted flights for airport: LAX


In [5]:
cursor.execute("SELECT DISTINCT aircraft_registration FROM flights WHERE aircraft_registration IS NOT NULL")
registrations = [row[0] for row in cursor.fetchall()]

print(f"Found {len(registrations)} registrations")


Found 3077 registrations


In [6]:
import requests
import sqlite3
import time

# Connect to SQLite
conn_sql = sqlite3.connect("flightdb.db")
cursor = conn_sql.cursor()

# Step 1: Get distinct registrations from flights
cursor.execute("SELECT DISTINCT aircraft_registration FROM flights WHERE aircraft_registration IS NOT NULL")
registrations = [row[0] for row in cursor.fetchall()]

print("Found registrations:", registrations[:10])  # preview first 10

# API headers
headers = {
    "x-rapidapi-key": "c07681eddemsh68bef06446b6e44p1c266ajsncd7de52d3ae4",  
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
}

# Step 2: Fetch details for each registration
for reg in registrations:
    url = f"https://aerodatabox.p.rapidapi.com/aircrafts/reg/{reg}"
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"{reg}: Error {response.status_code} - {response.text}")
        time.sleep(2)
        continue

    aircraft_data = response.json()

    # Skip if API returned an error message
    if "message" in aircraft_data:
        print(f"{reg}: {aircraft_data['message']}")
        time.sleep(2)
        continue

    cursor.execute("""
    INSERT OR IGNORE INTO aircraft
    (registration, model, manufacturer, icao_type_code, owner)
    VALUES (?, ?, ?, ?, ?)
    """, (
        aircraft_data.get("reg"),            # ✅ FIXED: use "reg"
        aircraft_data.get("model"),
        aircraft_data.get("manufacturer"),
        aircraft_data.get("icaoTypeCode"),
        aircraft_data.get("owner")
    ))

    print(f"Inserted aircraft: {reg}")
    time.sleep(2)  # avoid API rate limits

conn_sql.commit()

Found registrations: ['VN-A697', 'VT-TQU', 'HS-LVQ', 'VT-BDN', 'HS-THX', 'VT-IOG', 'VT-NHI', 'VT-NHH', 'VT-IMS', 'VT-IQP']
Inserted aircraft: VN-A697
Inserted aircraft: VT-TQU
Inserted aircraft: HS-LVQ
Inserted aircraft: VT-BDN
Inserted aircraft: HS-THX
Inserted aircraft: VT-IOG
Inserted aircraft: VT-NHI
Inserted aircraft: VT-NHH
Inserted aircraft: VT-IMS
Inserted aircraft: VT-IQP
Inserted aircraft: B-LQH
Inserted aircraft: VT-IWU
Inserted aircraft: F-HTYM
Inserted aircraft: 9H-CXF
Inserted aircraft: VT-RTS
Inserted aircraft: D-AIXZ
Inserted aircraft: VT-ILJ
Inserted aircraft: VT-NHM
Inserted aircraft: 9M-MXD
Inserted aircraft: D-ABVM
Inserted aircraft: OK-TVR
Inserted aircraft: 4R-ANB
Inserted aircraft: VT-NCS
Inserted aircraft: VT-YBO
Inserted aircraft: A7-BEH


KeyboardInterrupt: 

In [7]:
# Count rows in airport table
cursor.execute("SELECT COUNT(*) FROM airport")
print("Airport rows:", cursor.fetchone()[0])

# Count rows in flights table
cursor.execute("SELECT COUNT(*) FROM flights")
print("Flights rows:", cursor.fetchone()[0])

# Count rows in aircraft table
cursor.execute("SELECT COUNT(*) FROM aircraft")
print("Aircraft rows:", cursor.fetchone()[0])

# Count rows in airport_delays table
cursor.execute("SELECT COUNT(*) FROM airport_delays")
print("Delays rows:", cursor.fetchone()[0])


Airport rows: 10
Flights rows: 15825
Aircraft rows: 25
Delays rows: 9


In [8]:
import pandas as pd

def show_table_structure(table_name):
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = cursor.fetchall()
    df = pd.DataFrame(columns, columns=["cid", "name", "type", "notnull", "dflt_value", "pk"])
    print(f"\nStructure of {table_name} table:")
    print(df)

# Run for each table
show_table_structure("flights")
show_table_structure("aircraft")
show_table_structure("airport")
show_table_structure("airport_delays")



Structure of flights table:
    cid                   name  type  notnull dflt_value  pk
0     0              flight_id  TEXT        0       None   1
1     1          flight_number  TEXT        0       None   0
2     2  aircraft_registration  TEXT        0       None   0
3     3            origin_iata  TEXT        0       None   0
4     4       destination_iata  TEXT        0       None   0
5     5    scheduled_departure  TEXT        0       None   0
6     6       actual_departure  TEXT        0       None   0
7     7      scheduled_arrival  TEXT        0       None   0
8     8         actual_arrival  TEXT        0       None   0
9     9                 status  TEXT        0       None   0
10   10           airline_code  TEXT        0       None   0

Structure of aircraft table:
   cid            name     type  notnull dflt_value  pk
0    0     aircraft_id  INTEGER        0       None   1
1    1    registration     TEXT        0       None   0
2    2           model     TEXT        0 

In [9]:
import sqlite3
import pandas as pd

# Connect to your SQLite DB
conn = sqlite3.connect("flightdb.db")


In [10]:
# Check flights table
pd.read_sql_query("SELECT COUNT(*) FROM flights;", conn)

# Preview some aircraft registrations in flights
pd.read_sql_query("SELECT DISTINCT aircraft_registration FROM flights LIMIT 10;", conn)

# Check aircraft table
pd.read_sql_query("SELECT COUNT(*) FROM aircraft;", conn)

# Preview aircraft table
pd.read_sql_query("SELECT * FROM aircraft LIMIT 5;", conn)


,aircraft_id,registration,model,manufacturer,icao_type_code,owner


In [11]:
#1) Show the total number of flights for each aircraft model, listing the model and its count.

cursor.execute("""
SELECT a.model, COUNT(*) AS total_number
FROM flights f
JOIN aircraft a 
  ON f.aircraft_registration = a.registration
GROUP BY a.model
ORDER BY total_number DESC
""")
print(cursor.fetchall())

[('A21N', 36), ('A20N', 10), ('B738', 9), ('A359', 9), ('B773', 2), ('B752', 2), ('B744', 2), ('B739', 1), ('B38M', 1)]


In [12]:
#2.List all aircraft (registration, model) that have been assigned to more than 5 flights.

import pandas as pd

cursor.execute("""
SELECT a.model, COUNT(*) AS total_number
FROM flights f
JOIN aircraft a 
  ON f.aircraft_registration = a.registration
GROUP BY a.model
ORDER BY total_number DESC
""")

rows = cursor.fetchall()
cols = [desc[0] for desc in cursor.description]
print(pd.DataFrame(rows, columns=cols))

  model  total_number
0  A21N            36
1  A20N            10
2  B738             9
3  A359             9
4  B773             2
5  B752             2
6  B744             2
7  B739             1
8  B38M             1


In [13]:
#3.For each airport, display its name and the number of outbound flights, but only for airports with more than 5 flights

import pandas as pd

cursor.execute("""
SELECT a.registration, a.model, COUNT(*) AS flight_count
FROM flights f
JOIN aircraft a 
  ON f.aircraft_registration = a.registration
GROUP BY a.registration, a.model
HAVING COUNT(*) > 5
ORDER BY flight_count DESC
""")

rows = cursor.fetchall()
cols = [desc[0] for desc in cursor.description]
print(pd.DataFrame(rows, columns=cols))

  registration model  flight_count
0       VT-NHH  A21N             7
1       OK-TVR  B738             6
2       VT-IMS  A21N             6
3       VT-IWU  A21N             6


In [14]:
#4Find the top 3 destination airports (name, city) by number of arriving flights, sorted by count descending
import pandas as pd

query_4 = """
SELECT 
    ap.name AS airport_name,
    CASE 
        WHEN ap.iata_code = 'DEL' THEN 'Delhi'
        WHEN ap.iata_code = 'BOM' THEN 'Mumbai'
        WHEN ap.iata_code = 'SIN' THEN 'Singapore'
        WHEN ap.iata_code = 'BLR' THEN 'Bengaluru'
        WHEN ap.iata_code = 'HYD' THEN 'Hyderabad'
        WHEN ap.iata_code = 'MAA' THEN 'Chennai'
        WHEN ap.iata_code = 'JFK' THEN 'New York'
        WHEN ap.iata_code = 'LHR' THEN 'London'
        WHEN ap.iata_code = 'SYD' THEN 'Sydney'
        WHEN ap.iata_code = 'LAX' THEN 'Los Angeles'
        ELSE COALESCE(ap.city, ap.name)
    END AS city,
    COUNT(*) AS arrivals_count
FROM flights f
JOIN airport ap ON f.destination_iata = ap.iata_code
GROUP BY ap.iata_code, ap.name, ap.city
ORDER BY arrivals_count DESC
LIMIT 3;
"""

print(pd.read_sql_query(query_4, conn))

      airport_name       city  arrivals_count
0         Heathrow     London            2264
1           Changi  Singapore            1769
2  Kingsford Smith     Sydney            1031


In [17]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

# Let's see what airports actually exist in your table right now
df_check = pd.read_sql_query("SELECT * FROM airport LIMIT 10;", conn)
conn.close()

print(f"Total airports found: {len(df_check)}")
display(df_check)

Total airports found: 10


,airport_id,icao_code,iata_code,name,city,country,continent,latitude,longitude,timezone
0,1,VOBL,BLR,Bengaluru,Bangalore,None,None,13.197899,77.706300,None
1,2,VABB,BOM,Chhatrapati Shivaji,Mumbai,None,None,19.088700,72.867900,None
2,3,VIDP,DEL,Indira Gandhi,New Delhi,None,None,28.566500,77.103100,None
3,4,VOMM,MAA,Chennai,Chennai,None,None,12.990005,80.169300,None
4,5,VOHS,HYD,Rajiv Gandhi,Hyderabad,None,None,17.231318,78.429855,None
5,6,KJFK,JFK,John F Kennedy,New York,None,None,40.639800,-73.778900,None
6,7,EGLL,LHR,Heathrow,London,None,None,51.470600,-0.461941,None
7,8,WSSS,SIN,Changi,Singapore,None,None,1.350190,103.994000,None
8,9,YSSY,SYD,Kingsford Smith,Sydney,None,None,-33.946100,151.177000,None
9,10,KLAX,LAX,Los Angeles,Los Angeles,None,None,33.942500,-118.408000,None


In [1]:
import sqlite3

# 1. Connect to your database
conn = sqlite3.connect("flightdb.db")
cursor = conn.cursor()

# 2. Update the 'country' column for your 10 airports manually
country_updates = [
    ("India", "BLR"),
    ("India", "BOM"),
    ("India", "DEL"),
    ("India", "MAA"),
    ("India", "HYD"),
    ("USA", "JFK"),
    ("UK", "LHR"),
    ("Singapore", "SIN"),
    ("Australia", "SYD"),
    ("USA", "LAX")
]

cursor.executemany("""
UPDATE airport 
SET country = ? 
WHERE iata_code = ?;
""", country_updates)

conn.commit()
conn.close()
print("✅ Database successfully patched! Missing country fields have been populated.")

✅ Database successfully patched! Missing country fields have been populated.


In [2]:
#5.Show for each flight: number, origin, destination, and a label 'Domestic' or International' using CASE WHEN on country match
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

query = """
SELECT 
    f.flight_number,
    f.origin_iata,
    f.destination_iata,
    CASE 
        -- 1. If an unlisted airport code appears in your flights dataset, flag safely
        WHEN org_air.country IS NULL OR dest_air.country IS NULL THEN 'Unknown'
        
        -- 2. If both countries match exactly, it's Domestic
        WHEN org_air.country = dest_air.country THEN 'Domestic'
        
        -- 3. Otherwise it's International
        ELSE 'International'
    END AS flight_type
FROM 
    flights f
LEFT JOIN airport org_air ON f.origin_iata = org_air.iata_code
LEFT JOIN airport dest_air ON f.destination_iata = dest_air.iata_code;
"""

df_final = pd.read_sql_query(query, conn)
conn.close()

# View your beautifully corrected results!
display(df_final.head(15))

,flight_number,origin_iata,destination_iata,flight_type
0,VJ 1802,BLR,SGN,Unknown
1,AI 2758,BLR,DEL,Domestic
2,SL 217,BLR,DMK,Unknown
3,BZ 304,BLR,DEL,Domestic
4,TG 326,BLR,BKK,Unknown
5,6E 1605,BLR,DPS,Unknown
6,6E 839,BLR,DEL,Domestic
7,6E 5293,BLR,BOM,Domestic
8,6E 6563,BLR,PNQ,Unknown
9,6E 77,BLR,JED,Unknown


In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

query = """
SELECT 
    f.flight_number,
    f.origin_iata,
    f.destination_iata,
    CASE 
        -- 1. If both airports exist in our table and countries match, it's Domestic
        WHEN org_air.country IS NULL OR dest_air.country IS NULL THEN 'International'
        
        -- 2. If countries match exactly, it's Domestic
        WHEN org_air.country = dest_air.country THEN 'Domestic'
        
        -- 3. Anything else is cleanly International
        ELSE 'International'
    END AS flight_type
FROM 
    flights f
LEFT JOIN airport org_air ON f.origin_iata = org_air.iata_code
LEFT JOIN airport dest_air ON f.destination_iata = dest_air.iata_code;
"""

df_final = pd.read_sql_query(query, conn)
conn.close()

display(df_final.head(15))

,flight_number,origin_iata,destination_iata,flight_type
0,VJ 1802,BLR,SGN,International
1,AI 2758,BLR,DEL,Domestic
2,SL 217,BLR,DMK,International
3,BZ 304,BLR,DEL,Domestic
4,TG 326,BLR,BKK,International
5,6E 1605,BLR,DPS,International
6,6E 839,BLR,DEL,Domestic
7,6E 5293,BLR,BOM,Domestic
8,6E 6563,BLR,PNQ,International
9,6E 77,BLR,JED,International


In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

# This query pulls the actual country strings from your tables so you can verify the logic
verify_query = """
SELECT 
    f.flight_number,
    f.origin_iata,
    org_air.country AS origin_country,
    f.destination_iata,
    dest_air.country AS destination_country,
    CASE 
        WHEN org_air.country IS NULL OR dest_air.country IS NULL THEN 'Missing Airport Data'
        WHEN org_air.country = dest_air.country THEN 'Domestic'
        ELSE 'International'
    END AS logic_check
FROM 
    flights f
LEFT JOIN airport org_air ON f.origin_iata = org_air.iata_code
LEFT JOIN airport dest_air ON f.destination_iata = dest_air.iata_code
LIMIT 15;
"""

df_verify = pd.read_sql_query(verify_query, conn)
conn.close()
display(df_verify)

,flight_number,origin_iata,origin_country,destination_iata,destination_country,logic_check
0,VJ 1802,BLR,India,SGN,NaN,Missing Airport Data
1,AI 2758,BLR,India,DEL,India,Domestic
2,SL 217,BLR,India,DMK,NaN,Missing Airport Data
3,BZ 304,BLR,India,DEL,India,Domestic
4,TG 326,BLR,India,BKK,NaN,Missing Airport Data
5,6E 1605,BLR,India,DPS,NaN,Missing Airport Data
6,6E 839,BLR,India,DEL,India,Domestic
7,6E 5293,BLR,India,BOM,India,Domestic
8,6E 6563,BLR,India,PNQ,NaN,Missing Airport Data
9,6E 77,BLR,India,JED,NaN,Missing Airport Data


In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

query = """
SELECT 
    f.flight_number,
    f.origin_iata,
    f.destination_iata,
    CASE 
        -- 1. If both countries match exactly, it is a Domestic route
        WHEN org_air.country = dest_air.country THEN 'Domestic'
        
        -- 2. If countries don't match, OR if the destination airport data 
        -- falls outside our 10-airport master set, categorize it as International
        ELSE 'International'
    END AS flight_type
FROM 
    flights f
LEFT JOIN airport org_air ON f.origin_iata = org_air.iata_code
LEFT JOIN airport dest_air ON f.destination_iata = dest_air.iata_code;
"""

df_final_submission = pd.read_sql_query(query, conn)
conn.close()

# View your final assignment dataframe layout!
display(df_final_submission.head(15))

,flight_number,origin_iata,destination_iata,flight_type
0,VJ 1802,BLR,SGN,International
1,AI 2758,BLR,DEL,Domestic
2,SL 217,BLR,DMK,International
3,BZ 304,BLR,DEL,Domestic
4,TG 326,BLR,BKK,International
5,6E 1605,BLR,DPS,International
6,6E 839,BLR,DEL,Domestic
7,6E 5293,BLR,BOM,Domestic
8,6E 6563,BLR,PNQ,International
9,6E 77,BLR,JED,International


In [ ]:
#6.Show the 5 most recent arrivals at “DEL” airport including flight number, aircraft,departure airport name, and arrival time, ordered by latest arrival time first. Use JOINs to get the aircraft model and departure airport name from their respective tables.

import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

del_arrivals_query = """
SELECT 
    f.flight_number,
    f.aircraft_registration,
    org_air.name AS departure_airport_name,
    f.scheduled_arrival
FROM 
    flights f
LEFT JOIN 
    airport org_air ON TRIM(f.origin_iata) = TRIM(org_air.iata_code)
WHERE 
    TRIM(f.destination_iata) = 'DEL'
    AND f.scheduled_arrival IS NOT NULL
    AND f.scheduled_arrival != 'NaN'
ORDER BY 
    f.scheduled_arrival DESC
LIMIT 5;
"""

df_del_arrivals = pd.read_sql_query(del_arrivals_query, conn)
conn.close()

print("✈️ 5 Most Recent Arrivals at DEL:")
display(df_del_arrivals)

✈️ 5 Most Recent Arrivals at DEL:


,flight_number,aircraft_registration,departure_airport_name,scheduled_arrival
0,AC 6454,NaN,Heathrow,2026-04-05 07:36Z
1,TP 7024,NaN,Heathrow,2026-04-05 07:36Z
2,TP 7036,VT-TSP,Heathrow,2026-04-05 06:44Z
3,AI 2016,VT-TSP,Heathrow,2026-04-05 06:44Z
4,DL 5947,NaN,Heathrow,2026-04-05 03:20Z


In [ ]:
#7.Find all airports with no arriving flights (never used as a destination in flights table)
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

# The REAL Query 7: Finds airports NOT IN the destination list
query_seven = """
SELECT 
    a.iata_code,
    a.name,
    a.city,
    a.country
FROM 
    airport a
WHERE 
    a.iata_code NOT IN (
        SELECT DISTINCT destination_iata 
        FROM flights 
        WHERE destination_iata IS NOT NULL
    );
"""

df_q7 = pd.read_sql_query(query_seven, conn)
conn.close()

print("🔍 Real Query 7 Output:")
display(df_q7)

🔍 Real Query 7 Output:


,iata_code,name,city,country


In [15]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

# Verification: Count how many times each master airport appears as a destination
check_query = """
SELECT 
    a.iata_code,
    a.name,
    COUNT(f.flight_id) AS times_used_as_destination
FROM 
    airport a
LEFT JOIN 
    flights f ON TRIM(a.iata_code) = TRIM(f.destination_iata)
GROUP BY 
    a.iata_code;
"""

df_check = pd.read_sql_query(check_query, conn)
conn.close()

display(df_check)

,iata_code,name,times_used_as_destination
0,BLR,Bengaluru,361
1,BOM,Chhatrapati Shivaji,471
2,DEL,Indira Gandhi,714
3,HYD,Rajiv Gandhi,255
4,JFK,John F Kennedy,671
5,LAX,Los Angeles,731
6,LHR,Heathrow,2264
7,MAA,Chennai,204
8,SIN,Changi,1769
9,SYD,Kingsford Smith,1031


In [18]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

query_eight_final = """
SELECT 
    f.airline_code,
    SUM(CASE WHEN f.status IN ('Arrived', 'Departed') THEN 1 ELSE 0 END) AS normal_on_time_count,
    SUM(CASE WHEN f.status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_count,
    SUM(CASE WHEN f.status IN ('Canceled', 'CanceledUncertain') THEN 1 ELSE 0 END) AS cancelled_count,
    COUNT(*) AS total_flights
FROM 
    flights f
GROUP BY 
    f.airline_code
ORDER BY 
    total_flights DESC;
"""

df_q8 = pd.read_sql_query(query_eight_final, conn)
conn.close()

print("📊 Query 8: Real Flight Status Matrix per Carrier")
display(df_q8)

📊 Query 8: Real Flight Status Matrix per Carrier


,airline_code,normal_on_time_count,delayed_count,cancelled_count,total_flights
0,6E,1279,3,11,1752
1,AA,913,0,58,981
2,AI,836,0,7,849
3,BA,811,0,16,829
4,DL,692,0,58,771
...,...,...,...,...,...
229,K7,1,0,0,1
230,D0,1,0,0,1
231,CC,1,0,0,1
232,BT,1,0,0,1


In [20]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

query_nine = """
SELECT 
    f.flight_number,
    f.aircraft_registration,
    org_air.name AS origin_airport,
    dest_air.name AS destination_airport,
    f.scheduled_departure
FROM 
    flights f
LEFT JOIN airport org_air ON TRIM(f.origin_iata) = TRIM(org_air.iata_code)
LEFT JOIN airport dest_air ON TRIM(f.destination_iata) = TRIM(dest_air.iata_code)
WHERE 
    f.status IN ('Canceled', 'CanceledUncertain')
ORDER BY 
    f.scheduled_departure DESC;
"""

df_q9 = pd.read_sql_query(query_nine, conn)
conn.close()

print("❌ Query 9: Log of Cancelled Flights")
display(df_q9)

❌ Query 9: Log of Cancelled Flights


,flight_number,aircraft_registration,origin_airport,destination_airport,scheduled_departure
0,DL 579,NaN,Los Angeles,NaN,2026-04-05 06:50Z
1,UA 4796,NaN,Los Angeles,NaN,2026-04-05 06:37Z
2,LY 1306,NaN,Los Angeles,NaN,2026-04-05 05:55Z
3,VS 24,NaN,Los Angeles,NaN,2026-04-05 05:30Z
4,AS 3325,NaN,Los Angeles,NaN,2026-04-05 05:20Z
...,...,...,...,...,...
416,DL 948,NaN,NaN,Los Angeles,NaN
417,DL 2202,NaN,NaN,Los Angeles,NaN
418,DL 449,NaN,NaN,Los Angeles,NaN
419,DL 2459,NaN,NaN,Los Angeles,NaN


In [21]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

query_ten = """
SELECT 
    f.origin_iata,
    f.destination_iata,
    COUNT(DISTINCT f.aircraft_registration) AS unique_aircraft_count
FROM 
    flights f
WHERE 
    f.aircraft_registration IS NOT NULL
GROUP BY 
    f.origin_iata, 
    f.destination_iata
HAVING 
    COUNT(DISTINCT f.aircraft_registration) > 2
ORDER BY 
    unique_aircraft_count DESC;
"""

df_q10 = pd.read_sql_query(query_ten, conn)
conn.close()

print("✈️ Query 10: Active City-Pairs Fleet Volume")
display(df_q10)

✈️ Query 10: Active City-Pairs Fleet Volume


,origin_iata,destination_iata,unique_aircraft_count
0,MEL,SYD,41
1,SYD,MEL,41
2,BLR,DEL,37
3,DEL,BLR,31
4,KUL,SIN,31
...,...,...,...
781,YUL,LHR,3
782,YYC,LAX,3
783,YYZ,DEL,3
784,ZRH,DEL,3


In [22]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flightdb.db")

query_eleven = """
SELECT 
    f.destination_iata AS destination_airport,
    SUM(CASE WHEN f.status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_arrivals,
    COUNT(*) AS total_arrivals,
    ROUND(
        SUM(CASE WHEN f.status = 'Delayed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        2
    ) AS delay_percentage
FROM 
    flights f
GROUP BY 
    f.destination_iata
ORDER BY 
    delay_percentage DESC;
"""

df_q11 = pd.read_sql_query(query_eleven, conn)
conn.close()

print("⏳ Query 11: Final Arrival Delay Percentages")
display(df_q11)

⏳ Query 11: Final Arrival Delay Percentages


,destination_airport,delayed_arrivals,total_arrivals,delay_percentage
0,BLR,9,361,2.49
1,BOM,2,471,0.42
2,ZYL,0,1,0.00
3,ZRH,0,43,0.00
4,ZQN,0,5,0.00
...,...,...,...,...
544,ABX,0,4,0.00
545,ABV,0,5,0.00
546,ABQ,0,7,0.00
547,ABJ,0,1,0.00


In [24]:
import sqlite3
import pandas as pd

# 1. Connect to the database file Streamlit is looking for
# Make sure this file matches the path in your app.py folder!
conn_streamlit = sqlite3.connect("flightdb.db")

# 2. Export your working DataFrames directly into SQL tables
# Replace 'df_flights', 'df_aircraft', and 'df_airports' with the actual variable names used in your notebook
try:
    df_flights.to_sql("flights", conn_streamlit, if_exists="replace", index=False)
    print("✅ Successfully exported flights data!")
except NameError:
    print("❌ Could not find df_flights. Check your notebook variable name.")

try:
    df_aircraft.to_sql("aircraft", conn_streamlit, if_exists="replace", index=False)
    print("✅ Successfully exported aircraft data!")
except NameError:
    print("❌ Could not find df_aircraft. Check your notebook variable name.")

try:
    df_airports.to_sql("airport", conn_streamlit, if_exists="replace", index=False)
    print("✅ Successfully exported airport data!")
except NameError:
    print("❌ Could not find df_airports. Check your notebook variable name.")

# 3. Commit changes and close connection safely
conn_streamlit.commit()
conn_streamlit.close()
print("🎉 All tables written! Refresh your Streamlit browser tab now.")

❌ Could not find df_flights. Check your notebook variable name.
❌ Could not find df_aircraft. Check your notebook variable name.
❌ Could not find df_airports. Check your notebook variable name.
🎉 All tables written! Refresh your Streamlit browser tab now.
